# Леਸਨ 05 - ਏਜੈਂਟਿਕ RAG


## Setup

ਇਹ ਨੋਟਬੁੱਕ ਮਾਈਕ੍ਰੋਸੋਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ ਏਜੈਂਟਿਕ RAG (ਰੇਟ੍ਰੀਵਲ-ਆਗਮੈਂਟਡ ਜੈਨਰੇਸ਼ਨ) ਪੈਟਰਨ ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ।

**ਪੂਰਵਸ਼ਰਤਾਂ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ਤੁਹਾਡਾ Azure AI ਖੋਜ ਸੇਵਾ ਏਂਡਪੌਇੰਟ
- `AZURE_SEARCH_API_KEY` — ਤੁਹਾਡਾ Azure AI ਖੋਜ API ਕੁੰਜੀ
- ਵਾਤਾਵਰਨ ਚਲਾਉਣ ਵਾਲੇ ਪਰਿਵਰਤਨਾਂ ਰਾਹੀਂ ਸੰਰਚਿਤ Azure OpenAI ਤैनਾਤੀ
- Azure CLI ਪ੍ਰਮਾਣਿਤ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਏਜੇਂਟਿਕ RAG ਕੀ ਹੈ?

ਪारੰਪਰਿਕ RAG ਇੱਕ ਨਿਸ਼ਚਿਤ ਪਾਈਪਲਾਈਨ ਦੀ ਪਾਲਣਾ ਕਰਦਾ ਹੈ: ਦਸਤਾਵੇਜ਼ ਪ੍ਰਾਪਤ ਕਰੋ, ਫਿਰ ਜਵਾਬ ਤਿਆਰ ਕਰੋ। **ਏਜੇਂਟਿਕ RAG** ਇਸ ਤੋਂ ਅੱਗੇ ਵੱਧਦਾ ਹੈ ਜਿਸ ਨਾਲ ਏਜੰਟ ਨੂੰ ਇਸ ਗੱਲ ਦੀ ਆਜ਼ਾਦੀ ਦਿੱਤੀ ਜਾਂਦੀ ਹੈ ਕਿ ਉਹ **ਕਦੋਂ** ਅਤੇ **ਕਿਵੇਂ** ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰੇ।

ਏਜੈਂਟਿਕ RAG ਨਾਲ, ਏਜੰਟ:
- **ਫੈਸਲਾ ਕਰ ਸਕਦਾ ਹੈ** ਕਿ ਸawaਲ ਦੇ ਜਵਾਬ ਤੋਂ ਪਹਿਲਾਂ ਪ੍ਰਾਪਤੀ ਦੀ ਲੋੜ ਹੈ ਜਾਂ ਨਹੀਂ
- **ਚੁਣ ਸਕਦਾ ਹੈ** ਕਿ ਕਿਹੜੇ ਡੇਟਾ ਸਰੋਤ ਜਾਂ ਟੂਲ ਨੂੰ ਪੁੱਛਣਾ ਹੈ
- **ਮੂਲਾਂਕਣ ਕਰ ਸਕਦਾ ਹੈ** ਪ੍ਰਾਪਤ ਕੀਤੇ ਨਤੀਜਿਆਂ ਦਾ ਅਤੇ ਜੇ ਪਹਿਲਾ ਯਤਨ ਅਪੂਰਣ ਹੈ ਤਾਂ ਅੱਗੇ ਵੱਧ ਕੇ ਹੋਰ ਪ੍ਰਾਪਤੀਆਂ ਕਰ ਸਕਦਾ ਹੈ
- **ਜੋੜ ਸਕਦਾ ਹੈ** ਕਈ ਪ੍ਰਾਪਤੀ ਕਦਮਾਂ ਵਿੱਚੋਂ ਮਿਲੀ ਜਾਣਕਾਰੀ ਨੂੰ ਇੱਕ ਸੰਗਠਿਤ ਜਵਾਬ ਵਿੱਚ

ਇਸ ਨਾਲ ਏਜੰਟ ਇੱਕ ਅਥਾਹ ਤੇ ਸਹੀ ਪ੍ਰਾਪਤੀ-ਫਿਰ-ਜਨਰੇਟ ਪਾਈਪਲਾਈਨ ਨਾਲੋਂ ਵੱਧ ਲਚਕੀਲਾ ਅਤੇ ਸਹੀ ਬਣ ਜਾਂਦਾ ਹੈ।


## Creating a Search Tool

In Agentic RAG, external data sources are wrapped as **tools** that the agent can invoke on demand. This lets the agent treat retrieval as just another action it can take, rather than a mandatory step.

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ ਯਾਤਰਾ ਗਿਆਨ ਅਧਾਰ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ ਅਤੇ ਇਸਨੂੰ ਇੱਕ ਸੰਦ ਵਜੋਂ ਪ੍ਰਗਟ ਕਰਦੇ ਹਾਂ ਜਿਸਨੂੰ ਏਜੰਟ ਟਰਗੇਟ ਜਾਣਕਾਰੀ ਵੇਖਣ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ਏਜੰਟ ਬਣਾਉਣਾ

ਹੁਣ ਅਸੀਂ ਇੱਕ ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜੋ **ਜਵਾਬ ਦੇਣ ਤੋਂ ਪਹਿਲਾਂ ਹਮੇਸ਼ਾ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰਨ ਲਈ ਹੁਕਮਿਤ ਕੀਤਾ ਗਿਆ ਹੈ**। ਏਜੰਟ ਆਪਣੇ ਜਵਾਬਾਂ ਨੂੰ ਆਪਣੇ ਖੁਦ ਦੇ ਟ੍ਰੇਨਿੰਗ ਡਾਟਾ 'ਤੇ ਨਿਰਭਰ ਕਰਨ ਦੀ ਬਜਾਏ, ਗਿਆਨ ਬੇਸ ਵਿੱਚ ਅਧਾਰਿਤ ਕਰਨ ਲਈ `search_travel_knowledge` ਟੂਲ ਵਰਤਦਾ ਹੈ।


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterative Retrieval — The Maker-Checker Pattern

A key advantage of Agentic RAG is **iterative retrieval**. The agent can perform multiple rounds of search to verify, refine, or expand on its initial findings — similar to a "maker-checker" workflow:

1. **Maker step**: The agent retrieves initial information and drafts a response.
2. **Checker step**: The agent performs additional retrievals to verify details or fill gaps.

Below, the agent is asked a question that requires comparing multiple destinations, prompting it to search several times.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਕੇ **Agentic RAG** ਸਿਸਟਮ ਕਿਵੇਂ ਬਣਾਇਆ ਜਾ ਸਕਦਾ ਹੈ:

- **Agentic RAG** ਏਜੰਟਸ ਨੂੰ ਸੁਤੰਤਰ ਤੌਰ 'ਤੇ ਇਹ ਫੈਸਲਾ ਕਰਨ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ ਕਿ ਜਾਣਕਾਰੀ ਕਦੋਂ ਪ੍ਰਾਪਤ ਕਰਨੀ ਹੈ, ਜਿਸ ਨਾਲ ਪ੍ਰਾਪਤੀ ਇੱਕ ਨਿਰਧਾਰਿਤ ਨਹੀਂ ਬਲਕਿ ਗਤੀਸ਼ੀਲ ਬਣ ਜਾਂਦੀ ਹੈ।
- **ਉਪਕਾਰਣਾਂ ਨੂੰ ਜਾਣਕਾਰੀ ਸਰੋਤ ਵਜੋਂ**: ਬਾਹਰੀ ਗਿਆਨ ਆਧਾਰ (ਜਿਵੇਂ Azure AI Search) ਨੂੰ ਉਪਕਾਰਣਾਂ ਵਜੋਂ ਲਪੇਟਿਆ ਜਾਂਦਾ ਹੈ, ਜਿਨ੍ਹਾਂ ਨੂੰ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।
- **ਦੋਹਰਾਈ ਪ੍ਰਾਪਤੀ**: ਮੈਕਰ-ਚੈੱਕਰ ਪੈਟਰਨ ਏਜੰਟ ਨੂੰ ਕਈ ਵਾਰੀ ਖੋਜ, ਜਾਂਚ ਅਤੇ ਸੁਧਾਰ ਕਰਨ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ ਤਾ ਕਿ ਅੰਤਿਮ ਉੱਤਰ ਤਿਆਰ ਕੀਤਾ ਜਾ ਸਕੇ।

ਉਤਪਾਦਨ ਵਿੱਚ, ਤੁਸੀਂ ਇਨ-ਮੇਮੋਰੀ `TRAVEL_KNOWLEDGE_BASE` ਨੂੰ ਅਸਲ Azure AI Search ਇੰਡੈਕਸ ਨਾਲ ਬਦਲੋਗੇ ਤਾਂ ਜੋ ਵੱਡੇ ਪੈਮਾਨੇ 'ਤੇ ਯਾਤਰਾ ਦਸਤਾਵੇਜ਼ਾਂ ਦੀ ਪ੍ਰਾਪਤੀ ਕੀਤੀ ਜਾ ਸਕੇ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
